# Engineering Lab Log · Session 17: Prompt Evaluation
## Automated LLM-as-Judge, Golden Sets, and Multi-Criteria Rubrics

**Lab Structure & Execution Architecture**

| Module Part | Scope & Purpose | Execution Mode |
|---|---|---|
| **Part A · Architectural Foundations** | Systematic evaluation theory, bias mitigations, and pipeline topology | Theoretical grounding & system definitions |
| **Part B · Empirical Benchmarking** | Sections 1 to 9: Triage, single-judge testing, baseline scoring, structured comparison, human calibration, report serialization | Active test pipeline execution |
| **Part C · Synthesis & Auditing** | Production trade-offs, inter-rater reliability analysis, CI/CD verification | Engineering conclusions & review |

## 1. Architectural Motivation: Multi-Criteria Quality Evaluation

### The Evaluation Gap: Beyond Single-Label Accuracy
Session 16 established structured prompt engineering, demonstrating an improvement from 13/20 (Session 14 baseline) to 18/20 on held-out ticket classification. However, evaluating customer-facing support pipelines on category labels alone obscures critical production risks:
- **Policy Hallucination:** A completion can assign the correct category (`account`) while inventing binding promises (e.g. unauthorized fee waivers or cooling-off refund windows).
- **Format Non-Compliance:** Unstructured completions that omit required reference IDs or prepend conversational preambles break downstream parsing automation.
- **Rule Incoherence:** The model may select the correct label while citing contradictory reasoning or fictitious house rules.

Relying on ad-hoc manual review does not scale across version iterations and introduces reviewer fatigue. This lab operationalizes the second half of a governed LLM engineering pipeline: an immutable **golden test set** (`GOLDEN20`), a pre-committed **multi-criteria evaluation rubric** (`c1`–`c4`), and an automated **LLM-as-judge** returning strict JSON payloads.

### Quality Governance Framework

| Engineering Guardrail | Failure Mode Prevented | Architectural Implementation |
|---|---|---|
| **Pre-Committed Rubric** | Retroactive target-shifting (grading toward whatever output looks appealing) | 4 discrete boolean checks (`c1`–`c4`) defined prior to completion review |
| **Strict JSON LLM Judge** | Ambiguous conversational critiques that cannot be automated in CI | Forced `{c1: PASS/FAIL, ...}` schema with regex extraction and key validation |
| **Deterministic Order Guard** | Position bias, socket connection warming, and provider prompt-prefix caching | Deterministic hashing (`shuffle_order`) alternating prompt execution sequence |
| **Human Calibration Sample** | Unchecked model drift and systematic judge hallucination | 5-ticket spot-check benchmarking human vs. judge agreement rate |

### Enterprise Production Context
- **Regulated Industries (Banking / Healthcare):** Model risk governance mandates auditable, deterministic benchmarks showing non-regression before deploying prompt revisions to production.
- **A/B Testing Safeguards:** Offline automated evaluation on golden corpora filters out regression risks before exposing candidate prompts to live user traffic.

### Curriculum Progression & Technical Lineage

```
 S14  Prompt anatomy ──────────► Unstructured single-line baseline ("v1")
  │                              
 S15  Reasoning prompts ───────► DiskCache (zero-cost replay) and Meter (token accounting)
  │
 S16  Structured prompting ────► XML tags + Jinja2 template + versioned system.md ("v2")
  │
 S17  Prompt evaluation ───────► CURRENT: Golden corpus, 4-point rubric, LLM judge (v1 vs v2)
  │
 S18  Defensive prompting ─────► Adversarial jailbreak attacks evaluated with Session 17 judge harness
 S19  Chat architecture ───────► Stateful multi-turn assistant contracts
 S53–54  LLMOps ───────────────► Continuous automated prompt regression testing in CI/CD pipelines
```

**Runtime Dependencies:** Pre-configured environment credentials via Groq client, cached token ledger, and versioned prompts (`prompts/support_classifier/system.md`, `user.j2`) evaluated as immutable benchmark inputs.

### Core Evaluation Lexicon & Conceptual Taxonomy

---
**1. Golden set**
- **Definition:** a fixed, agreed list of test cases, kept exactly as-is across every prompt you later compare.
- **Analogy:** the official answer key for an exam — written once, not edited to fit whoever is sitting the exam.
- **Example:** today's `GOLDEN20` is *exactly* Session 16's `HELDOUT` — the same twenty tickets, same order, same labels.
- **Often confused with:** a held-out set. They can be the same data; the name changes with the job. "Held-out" means *not used as a training/few-shot example*. "Golden" means *the fixed set we score every version against*.
- **Comes back in:** S18, S53–54.

---
**2. Rubric**
- **Definition:** a written list of pass/fail checks, agreed **before** you read any answers.
- **Analogy:** a marking scheme a teacher writes before the exam, not after seeing who wrote what.
- **Example:** today's four criteria — correct label, cites the right house rule, follows the reply format, invents no policy.
- **Often confused with:** a 1-to-10 quality score. A score invites "it felt like an 8"; a rubric forces a yes/no per concrete check.
- **Comes back in:** S53–54.

---
**3. LLM-as-judge**
- **Definition:** using a second model call to grade the first model's output against a rubric.
- **Analogy:** a second pair of eyes marking the same exam with the same marking scheme.
- **Example:** today's `judge_answer()` sends one ticket, one answer and the rubric to the model and asks for PASS/FAIL per criterion.
- **Often confused with:** the model grading *itself* in the same call. Today's answerer and judge are separate calls (even when they happen to be the same underlying model).
- **Comes back in:** S18 (re-measuring after an attack).

---
**4. Strict-JSON verdict**
- **Definition:** forcing the judge's reply into a fixed shape (exactly four named keys, each PASS or FAIL) so code can total it up.
- **Analogy:** a bubble-sheet answer form instead of an essay — harder to mis-read, easy to machine-score.
- **Example:** `{"c1": "PASS", "c2": "FAIL", "c3": "PASS", "c4": "PASS"}`
- **Often confused with:** asking for a written explanation as well. You can ask for one, but never let the score depend on parsing free text.
- **Comes back in:** S20 (function calling asks for structure the same way, for a different reason).

---
**5. Judge bias — position bias**
- **Definition:** a judge's tendency to prefer whichever answer it reads first (or in a fixed, predictable slot), regardless of quality.
- **Analogy:** a recruiter who, without noticing, rates the first candidate of the day more kindly.
- **Example:** always answering v1 before v2, for every ticket, could make v1 look stronger for no reason but order.
- **Often confused with:** a real quality difference. The fix (section 11) is to vary the order, then see if the gap survives.
- **Comes back in:** S18.

---
**6. Judge bias — verbosity bias**
- **Definition:** a judge's tendency to reward longer, more confident-sounding replies, even when a short one is objectively correct.
- **Analogy:** grading an exam answer higher just because it filled the page.
- **Example:** a padded reply that still gets the label wrong scoring better than a terse, correct one.
- **Often confused with:** thoroughness actually being asked for. The fix is a pass/fail rubric rather than an open "rate the quality" scale, which resists this better (not perfectly).
- **Comes back in:** —

---
**7. Judge bias — self-preference bias**
- **Definition:** a model rating output that resembles its own typical phrasing or reasoning style more highly.
- **Analogy:** a teacher who was once a journalist marking essays written in journalistic style more kindly.
- **Example:** today's answerer and judge are the same model family — a real limit of this lab, named honestly rather than hidden.
- **Often confused with:** a factual quality judgement. The fix named on the guardrails slide — a different, ideally stronger, model as judge — is not exercised in this notebook; it is a known next step.
- **Comes back in:** —

---
**8. Order guard**
- **Definition:** deterministically varying, per test item, which of two things happens first, so no fixed ordering can bias the result.
- **Analogy:** alternating who serves first in a match, rather than always the same player.
- **Example:** `shuffle_order()` decides per ticket whether v1 or v2 is answered and judged first — the same choice every run (no hidden randomness), but not always the same prompt.
- **Often confused with:** true randomness. It is a deterministic hash, so your report is reproducible.
- **Comes back in:** —

---
**9. Human spot-check**
- **Definition:** grading a small sample yourself, with the same rubric, and comparing with the judge's verdicts.
- **Analogy:** a teacher re-marking five exam papers from a batch a teaching assistant marked, to check the TA's marking is sound.
- **Example:** today you grade five tickets by hand and compute an agreement rate with the judge.
- **Often confused with:** proof the judge is right everywhere. Five tickets is evidence, not a guarantee, for every ticket the judge ever grades.
- **Comes back in:** —

---
**10. Pass rate**
- **Definition:** the fraction of graded items that passed a given criterion (or passed every criterion at once).
- **Analogy:** the percentage of students who passed a particular exam question.
- **Example:** `16/20 = 80%` for "correct label".
- **Often confused with:** accuracy in the Session 16 sense. Session 16's accuracy was one criterion (the label) measured directly. Today's pass rate is one of *four* rubric criteria, each graded by the judge, not by exact string comparison.
- **Comes back in:** S53–54.

### Evaluation Pipeline Architecture & Artifact Topology

### The big picture

```
┌──────────────────────────── COURSE REPOSITORY (you pull it; you do not edit it) ───────────────────────────┐
│                                                                                                            │
│  module02/structured_tasks.py (S16)   HELDOUT = 20 labelled tickets, 5 per label                           │
│     evaluation_tasks.py (today)   THE DATA                                                                  │
│     GOLDEN20 = structured_tasks.HELDOUT, reused, not rebuilt       RUBRIC = 4 pass/fail criteria             │
│     V1_INSTRUCTION = Session 14's baseline        JUDGE_SYSTEM / JUDGE_USER = the judge's own prompt         │
│                                                                                                            │
│  module02/evaluation_utils.py (today)   THE TOOLS (helpers this notebook calls)                             │
│     judge_messages · parse_verdict · judge_answer · score_prompt · compare_prompts · shuffle_order ·        │
│     spot_check_items · agreement · eval_markdown · check_s17_work                                           │
│                                                                                                            │
│  module02/structured_utils.py (S16)   load_prompt_file · pick_examples · structured_maker — v2 is built     │
│                                        from YOUR files using these same Session 16 helpers                  │
│  module02/reasoning_utils.py (S15)    Lab = talks to the model · DiskCache = free re-runs · Meter = tokens  │
│  module01/providers_utils.py (S11)    reads your .env and makes the Groq client                             │
│                                                                                                            │
│  module02/s17_evaluation.ipynb  ◄── YOU ARE HERE                                                            │
│  tests/test_s17.py               the checkpoint (pytest)                                                    │
└────────────────────────────────────────────────────────────────────────────────────────────────────────────┘
                                   │ the notebook reads your S16 files, writes a new report ▼
┌──────────────────────────── YOUR WORK REPOSITORY (yours; you commit and push it) ──────────────────────────┐
│  prompts/support_classifier/system.md   YOUR Session 16 file — read today, not changed                      │
│  prompts/support_classifier/user.j2     YOUR Session 16 file — read today, not changed                      │
│  experiments/s17_eval.md                NEW today: pass rates for v1 and v2, plus your spot-check            │
└────────────────────────────────────────────────────────────────────────────────────────────────────────────┘
```

### The journey of one ticket, today
```
 one GOLDEN20 ticket
        │
        ├──► answered by v1 (S14 baseline) ──► judged against the rubric ──► 4 x PASS/FAIL
        │
        └──► answered by v2 (YOUR S16 files) ──► judged against the rubric ──► 4 x PASS/FAIL
                                                       │
                                          which answer is judged FIRST is shuffled, per ticket
```

### Every helper this notebook calls

| Helper | Lives in | You give it | It gives back | Used in section |
|---|---|---|---|---|
| `v1_messages` | `evaluation_utils` | a ticket | messages for the S14 baseline | 5 |
| `v2_messages_from_files` | `evaluation_utils` | your work repo, a ticket | messages built from YOUR system.md + user.j2 | 6 |
| `answer_with` | `evaluation_utils` | a Lab, messages | the model's reply text | 5, 6 |
| `judge_messages` | `evaluation_utils` | ticket, truth, answer | messages for the judge | 4 |
| `parse_verdict` | `evaluation_utils` | the judge's reply text | `{"c1": True/False, ...}` | 4 |
| `judge_answer` | `evaluation_utils` | a Lab, ticket, truth, answer | one graded verdict | 4, 5, 6 |
| `shuffle_order` | `evaluation_utils` | the golden set | which prompt goes first, per ticket | 6 |
| `score_prompt` | `evaluation_utils` | answer Lab, judge Lab, a messages-builder | pass rates for one prompt | 5 |
| `compare_prompts` | `evaluation_utils` | both Labs, both messages-builders | pass rates for v1 AND v2, order-guarded | 6 |
| `spot_check_items` | `evaluation_utils` | the golden set | 5 tickets, deterministic | 7 |
| `agreement` / `agreement_rate` | `evaluation_utils` | human and judge verdicts | how often they agree | 7 |
| `eval_markdown` | `evaluation_utils` | both results, models, spot-check | the report text | 8 |
| `save_eval_report` / `check_s17_work` | `evaluation_utils` | your work repo | save the report / check your submission | 8 |

### The order of the sections

| # | Section | Calls the model? | Minutes |
|---|---|---|---|
| 1 | Setup | No | 3 |
| 2 | Predict first: two replies, which is better? | No | 5 |
| 3 | The golden set and the rubric | No | 4 |
| 4 | Write and test the judge on one ticket | Yes (1 answer + 1 judge call) | 6 |
| 5 | Score v1 (baseline) on all 20 | Yes (20 + 20 calls) | 7 |
| 6 | Score v2 (structured) on all 20, compare | Yes (20 + 20 calls) | 8 |
| 7 | Human spot-check (5 tickets) | No (uses section 5–6's saved rows) | 6 |
| 8 | Save your evaluation report, check your submission | No | 4 |
| 9 | Commit, and run the checkpoint | No (shell + pytest) | 3 |

## A5. What you will be able to do, and what you will submit

**By the end of this session you will be able to:**
1. Explain why a label match is not the same as a good reply, with a concrete example.
2. Write a rubric of pass/fail criteria before reading any answers.
3. Build a judge prompt that returns strict, parseable JSON, and parse it defensively.
4. Compare two prompt versions on the same golden set with the same judge, guarded against ordering bias.
5. Name three known biases of LLM-as-judge and the guardrail for each.
6. Grade a small sample by hand and compute agreement with the judge.

**What you submit (in your work repository):**

| File | What it must contain |
|---|---|
| `prompts/support_classifier/system.md`, `user.j2` | Unchanged from Session 16 — today reads them, does not edit them |
| `experiments/s17_eval.md` | Pass rates for v1 AND v2 across all four criteria, all twenty golden tickets, plus a five-ticket human spot-check |

**The checkpoint:** `uv run pytest tests/test_s17.py -rs` with `WORK_REPO` set. *A skipped test is not a pass.*

**Budget:** today makes roughly twice as many model calls as Session 16, because every ticket is answered AND judged, for both prompt versions: about 80 calls (20 tickets x 2 prompts x 2 — one answer call, one judge call). Expect **60,000–90,000** of your 200,000 daily tokens (an estimate; the meter shows your real number). Re-running an unchanged cell is free.

---
# Part B · Hands-on

## 1. Setup & Environment Initialization

### Engineering Objective
Initialize the runtime environment, establish API connections, configure local disk caching, and prepare the evaluation pipeline. Before running any model inference or grading, verify all required utilities from Sessions 11, 14, 15, and 16 are available and that the repository paths are properly wired.

### Pipeline Architecture & Roles
- **Dual-Role Model (`Lab`):** Initialized a single `openai/gpt-oss-20b` client playing two isolated roles: first as the support classifier generating candidate replies, and second as an impartial judge grading those replies against the rubric.
- **Isolated Cache (`s17_cache.json`):** Bound inference to `~/.genai_course/s17_cache.json` to prevent Session 16 classification runs from contaminating new evaluation outputs.
- **Evaluation Target:** Loaded `GOLDEN20` (20 tickets) and the 4-point pass/fail `RUBRIC` (`c1`–`c4`).

In [2]:
import sys; sys.path[:0] = [".", "module01", "module02", "../module01", "../module02"]
import os
from pathlib import Path

try:
    from evaluation_tasks import (  # today
        DEFAULT_FORMAT_RULE,
        GOLDEN20,
        JUDGE_SYSTEM,
        JUDGE_USER,
        RUBRIC,
        RUBRIC_IDS,
        SAMPLE_REPLY_A,
        SAMPLE_REPLY_B,
        SAMPLE_TICKET,
        V1_INSTRUCTION,
    )
    from evaluation_utils import (  # today
        S17_CACHE,
        JudgeParseError,
        agreement,
        agreement_rate,
        answer_with,
        check_s17_work,
        compare_prompts,
        escape_judge_tags,
        eval_markdown,
        judge_answer,
        judge_messages,
        parse_eval_report,
        parse_spot_check,
        parse_verdict,
        pass_rates,
        pct,
        save_eval_report,
        score_prompt,
        shuffle_order,
        spot_check_items,
        v1_messages,
        v2_messages_from_files,
    )
    from providers_utils import groq_client  # Session 11
    from reasoning_utils import DiskCache, Lab, Meter  # Session 15
    from structured_tasks import (  # Session 16
        GUIDE,
        HELDOUT,
        LABELS,
        REFERENCE_SYSTEM,
        REFERENCE_USER,
    )
    from structured_utils import (  # Session 16
        SUPPORT_DIR,
        SYSTEM_FILE,
        USER_FILE,
        load_prompt_file,
        pick_examples,
    )
except ImportError as e:
    raise ImportError(f"{e}. This notebook uses files from Sessions 11, 14, 15, 16 and today: run 'git pull' in the "
                        "course folder, then restart the kernel.") from e

MODEL = os.getenv("MODEL")
WORK_REPO = os.getenv("WORK_REPO")
cache, meter = DiskCache(S17_CACHE), Meter()
client = None
if os.getenv("BASE_URL") and os.getenv("API_KEY") and MODEL:
    client = groq_client()
lab = Lab(client, MODEL, cache=cache, meter=meter) if client else None     # type: ignore # one Lab plays BOTH roles today: answerer and judge
ready = lab is not None

print(f"model: {MODEL} | ready: {ready}")
print(f"answers are remembered in: {S17_CACHE}")
print(f"golden set: {len(GOLDEN20)} tickets · rubric: {len(RUBRIC)} criteria {RUBRIC_IDS}")
print(f"WORK_REPO: {WORK_REPO or '(not set — section 6 will need it)'}")
if not ready:
    print("No key: sections 4-6 print 'skipped'. Sections 2, 3, 7, 8 need no model and still run.")

model: openai/gpt-oss-20b | ready: True
answers are remembered in: C:\Users\Gojira\.genai_course\s17_cache.json
golden set: 20 tickets · rubric: 4 criteria ['c1', 'c2', 'c3', 'c4']
WORK_REPO: D:\Projects\genai-course-work


### Verification & Runtime Environment
- **Model Endpoint:** `openai/gpt-oss-20b` connected via Groq (`ready: True`).
- **Session Isolation:** Cache mapped to `C:\Users\Gojira\.genai_course\s17_cache.json` for deterministic, zero-cost repeats.
- **Evaluation Dataset:** `GOLDEN20` (20 held-out tickets) loaded across 4 evaluation criteria (`c1`, `c2`, `c3`, `c4`).
- **Work Repo:** Configured to `D:\Projects\genai-course-work` to dynamically load `system.md` and `user.j2`.

### Concept Check
*Today one `lab` plays two different roles in different calls — the "answerer" and the "judge". Why is it still fair to call these two SEPARATE calls, even though they may use the same underlying model?*

### My Answer:
We should call them two SEPERATE calls because they ARE two SEPERATE API calls. We're using the same camera but with different lenses.

> **Engineering Note:** LLM endpoints are strictly stateless. Even when hitting the identical model weights, each API request generates an independent context window with its own distinct system instructions and conversation history. In the judging call, the prior model output is treated as untrusted text wrapped in XML data delimiters rather than model memory.

## 2. Qualitative Triage: Reply Comparison Baseline

### Engineering Objective
Examine how traditional binary classification metrics fail to capture production risks. By comparing two candidate replies against a single ticket without automated tooling, establish why multi-dimensional rubric evaluation is necessary.

### Case Study: UPI Transfer Complaint
- **Input Ticket:** Customer transfer of Rs 7,000 debited but unreceived by recipient.
- **Axes Evaluated:** Category classification (UPI = `account`), rule citation, and policy hallucination risks.

In [3]:
print("TICKET:", SAMPLE_TICKET)
print()
print("REPLY A:", SAMPLE_REPLY_A)
print()
print("REPLY B:", SAMPLE_REPLY_B)

TICKET: A UPI transfer of Rs 7,000 left my account yesterday but the person who was supposed to receive it says nothing has come in. Please help, this is urgent.

REPLY A: This looks like an account issue. I've logged it and a specialist will review your UPI transaction within 48 hours. Reference: AX-4471.

REPLY B: Sorry for the trouble with your card. Per our 24-hour cooling-off period, charges are reversible if you report within a day. I've flagged this as card.


### Triage Analysis & Findings
- **Reply A:** Correctly categorizes the UPI issue as `account`, states a realistic 48-hour specialist review window, and assigns a ticket reference without inventing promises.
- **Reply B:** Severe failure on multiple axes: misclassifies UPI as `card` and invents an unapproved 24-hour charge reversal rule.

### Concept Check
*What exactly is wrong with reply B, beyond the label? Name at least two separate problems.*

### My Answer:
First of all it doesn't look like a card issue, so that's a minus, secondly the model decided a reply that promises an imaginary policy, it hallucinated. It should've forwaded the ticket to a specialist instead.

> **Engineering Note:** In customer-facing support applications, hallucinating binding terms (like an unapproved refund window) is far more hazardous than a simple misclassification. A misrouted ticket is slightly delayed; an invented promise creates direct financial liability.

## 3. Dataset Integrity: The Golden Set and Evaluation Rubric

### Engineering Objective
Lock down the test fixtures and evaluation criteria prior to inspecting any model completions. Establishing an immutable dataset and strict rubric prevents retroactive grading adjustments (grading toward whichever output looks intuitively appealing).

### Evaluation Fixtures
- **Test Corpus (`GOLDEN20`):** Aliased directly from Session 16's `HELDOUT` split (20 tickets, balanced across `card`, `loan`, `account`, and `other`).
- **Pass/Fail Rubric (`c1`–`c4`):** Defines four discrete boolean checks:
  1. `c1` (Correct label): Identifies the exact target category.
  2. `c2` (Rule citation): Correctly references the governing house rule.
  3. `c3` (Format compliance): Adheres to target response structure (e.g. label first, concise reasoning).
  4. `c4` (Policy hallucination): Contains zero unverified policies, waivers, or service commitments.

In [4]:
print("GOLDEN20 is Session 16's HELDOUT:", GOLDEN20 is HELDOUT)
print(f"{len(GOLDEN20)} tickets, 5 per label: {LABELS}")
print()
for c in RUBRIC:
    print(f"{c['id']}. {c['name']}")
    print(f"   {c['test']}")

GOLDEN20 is Session 16's HELDOUT: True
20 tickets, 5 per label: ['card', 'loan', 'account', 'other']

c1. Correct label
   The reply names exactly one of: card, loan, account, other - and it is the label this ticket is actually filed under in the golden set.
c2. Cites the right house rule
   If the reply gives any reasoning, that reasoning points at the house rule that actually decides this ticket (see the bank's labelling guide), not a plausible-sounding rule that does not apply.
c3. Follows the reply format
   The reply obeys the shape its own system prompt asked for (for example: label first, one short reason, nothing else extra).
c4. Invents no policy
   The reply states no bank policy, rate, fee, waiver or promise that is not in the labelling guide. 'I have logged this' or 'a specialist will review it' is fine; a specific invented rule is not.


### Verification & Findings
- **Identity Check:** `GOLDEN20 is HELDOUT` evaluated to `True`, verifying object identity in memory.
- **Class Balance:** Exact 5 tickets per category across all four production labels.
- **Evaluation Criteria:** All 4 discrete checks loaded into memory ready for the judge pipeline.

### Concept Check
*Why does it matter that `GOLDEN20 is HELDOUT` returns `True` rather than the two lists merely looking the same?*

### My Answer:
`GOLDEN20 is HELDOUT` returning `TRUE` matters because that means `GOLDEN20` and `HELDOUT` are the same thing. `GOLDEN20 is HELDOUT` is not the same as `GOLDEN20 == HELDOUT`.

> **Engineering Note:** In Python, `a == b` checks value equality (whether two lists contain identical text), whereas `a is b` checks object identity (whether both variables point to the exact same memory address). By importing `HELDOUT` directly as `GOLDEN20` without copying or rewriting, we guarantee:
> 1. **Zero Data Leakage:** The model never saw these 20 tickets during few-shot prompt construction.
> 2. **Benchmark Continuity:** Session 17 measures prompt performance on the exact same baseline that Session 16 scored, preventing subtle evaluation drift.

## 4. Single-Ticket Evaluation Pipeline: Testing the Automated Judge

### Engineering Objective
Before deploying automated evaluation across the full 40-answer suite (20 golden tickets × 2 prompt variants), construct and unit-test the evaluation loop against a single known ticket-answer pair. Isolating the judge to a single call verifies tag delimiters, schema conformity, and parser defensive barriers without burning quota on invalid configurations.

### Architecture: One Call, 4-Way Categorical Evaluation, Strict JSON
```
 system: You are a strict grader. For EACH of the 4 criteria, output PASS or FAIL.
         Output ONLY JSON: {"c1": ..., "c2": ..., "c3": ..., "c4": ...}

 user:   <ticket> ... </ticket>
         <rubric> 1. Correct label: ...   2. Cites the right house rule: ...
                  3. Reply format: ...     4. Invents no policy: ... </rubric>
         <rulebook> the bank's house rules, so the judge can check criterion 2 itself </rulebook>
         <answer> the model's reply, being graded </answer>
```
All untrusted text (both customer inputs and candidate completions) is wrapped inside explicit XML tags (`<ticket>`, `<rubric>`, `<rulebook>`, `<answer>`), preventing prompt injection attacks where candidate replies try to dictate their own evaluation score.

### Pipeline Execution
1. `judge_messages(...)`: Compiles the system and user messages, sanitizing nested delimiters via tag escaping.
2. `lab.chat(...)`: Dispatches the payload through the reasoning lab.
3. `parse_verdict(...)`: Enforces defensive schema parsing, isolating the JSON payload and validating strict `c1`–`c4` boolean mappings.

In [5]:
test_ticket, test_truth = GOLDEN20[0]
print(f"ticket:  {test_ticket}")
print(f"truth:   {test_truth}")
print(f"answer:  {SAMPLE_REPLY_A}\n")

if ready:
    verdict = judge_answer(lab, test_ticket, test_truth, SAMPLE_REPLY_A)
    print("raw judge reply:", verdict["raw"])
    print("parsed verdicts:", verdict["verdicts"])
    print("all four passed:", verdict["all_pass"])
    if verdict["error"]:
        print("PARSE ERROR:", verdict["error"])
    print(meter.report())
else:
    print("skipped (no key in .env)")

ticket:  My credit card statement shows a purchase in another country while I have been at home.
truth:   card
answer:  This looks like an account issue. I've logged it and a specialist will review your UPI transaction within 48 hours. Reference: AX-4471.

raw judge reply: {"c1":"FAIL","c2":"FAIL","c3":"FAIL","c4":"FAIL"}
parsed verdicts: {'c1': False, 'c2': False, 'c3': False, 'c4': False}
all four passed: False
spent so far: 1 call (0 answered from memory), 1,019 tokens = about 1% of a 200,000-token free daily limit


### Baseline Test Analysis (4a)
- **Ticket Input:** Fraudulent foreign credit card transaction while customer remained home (target truth: `card`).
- **Target Answer (SAMPLE_REPLY_A):** Discusses UPI transaction handling and 48-hour review (intended for a different ticket).
- **Automated Judge Verdict:** Returned all four failures (`{"c1":"FAIL","c2":"FAIL","c3":"FAIL","c4":"FAIL"}`), proving the judge accurately flagged categorical mismatch (`c1`), irrelevant rule (`c2`), invalid format context (`c3`), and inappropriate review commitments (`c4`).
- **Telemetry:** Dispatched 1 call consuming 1,019 tokens (roughly 1% of the daily free tier ceiling).

### Defensive Parser Unit Testing (4b)
To verify that downstream evaluation does not silently corrupt PASS/FAIL counts due to conversational model preambles, `parse_verdict()` applies regex isolation (`re.search(r"\{.*\}", text, re.DOTALL)`) before JSON parsing.

In [11]:
import json, re

def my_parse_verdict(text, ids=("c1", "c2", "c3", "c4")):
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        raise ValueError("no JSON object found")
    obj = json.loads(m.group(0))
    out = {}
    for i in ids:
        if i not in obj:
            raise ValueError(f"missing key {i}")
        v = str(obj[i]).strip().upper()
        if v not in ("PASS", "FAIL"):
            raise ValueError(f"{i} is not PASS/FAIL: {obj[i]!r}")
        out[i] = v == "PASS"
    return out

sample = 'Sure, here you go:\n{"c1": "PASS", "c2": "FAIL", "c3": "PASS", "c4": "PASS"}\nhope that helps!'
mine, real = my_parse_verdict(sample), parse_verdict(sample)
print("mine:", mine)
print("real:", real)
assert mine == real == {"c1": True, "c2": False, "c3": True, "c4": True}
print("match: True")

mine: {'c1': True, 'c2': False, 'c3': True, 'c4': True}
real: {'c1': True, 'c2': False, 'c3': True, 'c4': True}
match: True


### Verification & Parser Integrity (4b)
- **Unit Test Assertion:** Both custom parser and production `parse_verdict()` successfully extracted the embedded JSON object from conversational wrapper text (`"Sure, here you go: ... hope that helps!"`), evaluating to `match: True`.
- **Fail-Safe Contract:** The parsing function enforces an exact schema (`c1`–`c4`), rejecting non-compliant payloads.

### Concept Check
*Why does the real `parse_verdict` check for EXTRA keys too, not just missing ones? What could an extra key signal?*

### My Answer:
Well the extra key signals hallucination

> **Engineering Note:** In structured evaluation, extra keys are a high-risk symptom of three potential failures:
> 1. **Hallucination / Instruction Drift:** The model begins inventing its own grading criteria, explanatory sidebands, or meta-commentary rather than following the strict rubric contract.
> 2. **Delimiter Breakout / Prompt Injection:** Untrusted candidate reply text may have leaked out of `<answer>` delimiters and coerced the model into adding auxiliary data fields (e.g. `{"c1": "PASS", ..., "override_note": "approved by admin"}`).
> 3. **Downstream Pipeline Corruption:** Unanticipated dictionary keys can break downstream schema validators, database ingest pipelines, or reporting aggregation scripts. Rejecting extra keys strictly preserves schema immutability.

## 5. Baseline Evaluation: Scoring Prompt v1 Across the Golden Corpus

### Engineering Objective
Benchmark Session 14's unstructured single-line baseline (`V1_INSTRUCTION`) across the complete 20-ticket held-out golden set (`GOLDEN20`). Evaluating all four rubric axes (`c1`–`c4`) simultaneously exposes how bare-bones classification prompts perform when subjected to multi-dimensional production compliance checks.

### Pipeline Execution Architecture
```
  [20 GOLDEN20 Tickets] 
           │
           ├──► Answer Generation: v1_messages (20 calls)
           └──► Compliance Grading: judge_messages (20 calls)
           │
  [Aggregated Scorecard: 40 Total Inference Dispatches]
```
The execution runs 40 total model calls (20 completions + 20 independent judge evaluations). Token consumption and runtime latency are monitored via `Meter`.

In [ ]:
if ready:
    v1_result = score_prompt(lab, lab, v1_messages, GOLDEN20)
    for c in RUBRIC:
        print(
            f"{c['name']:<28} {pct(v1_result['per_criterion'][c['id']], v1_result['n'])}"
        )
    print(f"{'All four criteria':<28} {pct(v1_result['all_pass'], v1_result['n'])}")
    print(f"graded: {v1_result['graded']}/{v1_result['n']}")
    print(meter.report())
else:
    print("skipped (no key in .env)")
    v1_result = None

Correct label                15/20 = 75%
Cites the right house rule   13/20 = 65%
Follows the reply format     19/20 = 95%
Invents no policy            20/20 = 100%
All four criteria            13/20 = 65%
graded: 20/20
spent so far: 41 calls (0 answered from memory), 23,422 tokens = about 12% of a 200,000-token free daily limit


### Empirical Evaluation Results (Prompt v1 Baseline)
- **Correct label (`c1`):** 15/20 (75%) — Consistent with Session 14 baseline accuracy.
- **Cites house rule (`c2`):** 13/20 (65%) — Unstructured prompts frequently omit or hallucinate rule references when unprompted.
- **Follows format (`c3`):** 19/20 (95%) — Basic single-turn compliance remained high on simple tickets.
- **Invents no policy (`c4`):** 20/20 (100%) — Zero unverified bank policies hallucinated in this run.
- **All Four Criteria (Compound Pass):** 13/20 (65%) — Exactly 13 tickets satisfied every compliance constraint simultaneously.
- **Resource Telemetry:** 41 cumulative calls consuming 23,422 tokens (~12% of daily quota).

### Concept Check
*Why is `All four criteria` almost always much lower than any single criterion on its own?*

### My Answer:
All four criteria requires a logical AND across all four checks. A reply only passes if c1, c2, c3, and c4 are ALL True simultaneously. Failing any single criterion fails the ticket, so the combined rate is strictly bounded by the weakest link and drops whenever failures happen on different tickets.

> **Engineering Note:** In multi-objective evaluation pipelines, overall quality is a conjunct (`all([c1, c2, c3, c4])`), not an arithmetic mean. The compound pass rate represents the intersection ($P(c_1 \cap c_2 \cap c_3 \cap c_4)$). If errors are distributed across different test cases (e.g. ticket A misses the rule while ticket B misses the label), compound pass rate plummets even when individual component metrics appear acceptable.

## 6. Comparative Evaluation: Prompt v1 (Baseline) vs Prompt v2 (Structured)

### Engineering Objective
Conduct a controlled side-by-side benchmark comparing Session 14's unstructured baseline against Session 16's structured production prompt (`prompts/support_classifier/system.md` and `user.j2`). Measure the real-world impact of few-shot demonstrations, XML tag encapsulation, and explicit house rule definitions across all four evaluation axes.

### Experimental Design & Order Guard Architecture
```
 [GOLDEN20 Tickets] 
         │
         ├──► Deterministic Hash: shuffle_order(ticket) ──► Shuffles Execution Order (v1 vs v2)
         │
         ├──► Answer Generation & Judging (v1 vs v2)
         │
 [Comparative Scorecard: Relative Delta per Rubric Criterion]
```
To eliminate systemic ordering artifacts (such as provider prefix caching, connection pooling latency, or pairwise judge position bias), `compare_prompts` alternates execution sequence per ticket via deterministic hash.

In [13]:
if ready:
    if not WORK_REPO:
        print(
            "skipped: WORK_REPO is not set. Add it to .env or set it in the shell, then restart the kernel."
        )
        cmp = None
    else:
        v2_maker = lambda t: v2_messages_from_files(WORK_REPO, t)
        cmp = compare_prompts(lab, lab, v1_messages, v2_maker, GOLDEN20)
        v1_result, v2_result = cmp["v1"], cmp["v2"]
        print(f"{'Criterion':<28} {'v1 (baseline)':<16} {'v2 (structured)':<16}")
        for c in RUBRIC:
            print(
                f"{c['name']:<28} {pct(v1_result['per_criterion'][c['id']], v1_result['n']):<16} "
                f"{pct(v2_result['per_criterion'][c['id']], v2_result['n']):<16}"
            )
        print(
            f"{'All four criteria':<28} {pct(v1_result['all_pass'], v1_result['n']):<16} "
            f"{pct(v2_result['all_pass'], v2_result['n']):<16}"
        )
        print(
            f"\nticket order was v1-first for {sum(cmp['order'])}/{len(cmp['order'])} tickets (shuffled, not all-v1-first)"
        )
        print(meter.report())
else:
    print("skipped (no key in .env)")
    cmp = None

Criterion                    v1 (baseline)    v2 (structured) 
Correct label                15/20 = 75%      20/20 = 100%    
Cites the right house rule   13/20 = 65%      16/20 = 80%     
Follows the reply format     19/20 = 95%      20/20 = 100%    
Invents no policy            20/20 = 100%     20/20 = 100%    
All four criteria            13/20 = 65%      16/20 = 80%     

ticket order was v1-first for 11/20 tickets (shuffled, not all-v1-first)
spent so far: 66 calls (55 answered from memory), 40,568 tokens = about 20% of a 200,000-token free daily limit


### Empirical Evaluation Results & Comparative Delta
```
Criterion                    v1 (baseline)    v2 (structured)    Delta
Correct label                15/20 = 75%      20/20 = 100%       +25%
Cites the right house rule   13/20 = 65%      16/20 = 80%        +15%
Follows the reply format     19/20 = 95%      20/20 = 100%       +5%
Invents no policy            20/20 = 100%     20/20 = 100%        0%

All four criteria            13/20 = 65%      16/20 = 80%        +15%
```
- **Order Guard Distribution:** `v1-first` for 11/20 tickets (balanced deterministic interleaving).
- **Execution Telemetry:** 66 calls recorded (55 served instantly from `DiskCache`), consuming 40,568 tokens (~20% of daily quota).

### Concept Check
*If `shuffle_order` were replaced with "always answer v1 first, then v2", and v2 then scored higher, could you be sure that was not partly a position-bias artifact? Why or why not?*

### My Answer:
to answer the question, what does it matter if v1 was tested first or v2? v1 was already tested previously, and it doesn't matter even if we test them in that order because the llm can't remember anything right? well unless the judge has context memory of it's previous tokens. Then it would get position bias.

> **Engineering Note:** In single-turn, independent API calls where each completion is graded in isolation, individual model calls are technically stateless. However, deterministic order shuffling remains an essential engineering safeguard for two critical reasons:
> 1. **Infrastructure & Provider-Side Caching:** Modern LLM APIs (and local inference gateways) implement prompt-prefix caching (KV caching), connection warming, and dynamic rate-limiting. Always running one variant second means it consistently benefits from warmed caches and established socket pools.
> 2. **Pairwise Judge Portability:** In pairwise evaluation (such as LMSYS Chatbot Arena or whenever a judge model compares candidate A and candidate B side-by-side in one prompt), LLMs exhibit severe position bias (frequently favoring Option A or whichever reply appears first). Building an order guard into the evaluation harness ensures the benchmarking methodology remains robust whether evaluation is conducted point-wise or pair-wise.

## 7. Quality Assurance: Human-in-the-Loop Spot-Check

### Engineering Objective
Validate the automated LLM judge against human domain expertise. An automated evaluation pipeline cannot be accepted on blind trust; calibrating judge output against a fixed sample of ground-truth reviews calculates inter-rater reliability and identifies systemic judge biases (such as lenient formatting passes or over-strict rule citations).

### Verification Architecture
```
 [Deterministic 5-Ticket Subset: spot_check_items()]
             │
             ├──► Human Engineering Audit (MY_VERDICTS: 5 tickets × 4 criteria = 20 labels)
             ├──► Automated Model Judge (judge_verdicts: 20 labels)
             │
 [Inter-Rater Concordance: agreement_rate(pairs)]
```

In [14]:
spot = spot_check_items()
v2_rows_by_ticket = {r["ticket"]: r for r in cmp["v2"]["rows"]} if cmp else {}

for i, (ticket, truth) in enumerate(spot, 1):
    row = v2_rows_by_ticket.get(ticket)
    print(f"[{i}] truth={truth}")
    print(f"    ticket: {ticket}")
    if row:
        print(f"    v2 answer: {row['answer']}")
        print(f"    judge said: {row['verdicts']}")
    else:
        print(
            "    (run section 6 first to have an answer + judge verdict to compare against)"
        )
    print()

# YOUR TURN: grade these five yourself, True = PASS, False = FAIL, using the section 3 rubric
MY_VERDICTS = [
    {
        "c1": True,
        "c2": True,
        "c3": True,
        "c4": True,
    },  # ticket 1 - edit these five dicts to your own judgement
    {"c1": True, "c2": True, "c3": True, "c4": True},  # ticket 2
    {"c1": True, "c2": True, "c3": True, "c4": True},  # ticket 3
    {"c1": True, "c2": True, "c3": True, "c4": True},  # ticket 4
    {"c1": True, "c2": True, "c3": True, "c4": True},  # ticket 5
]

if cmp:
    judge_verdicts = [v2_rows_by_ticket[t]["verdicts"] for t, _ in spot]
    pairs = list(zip(MY_VERDICTS, judge_verdicts))
    spot_result = agreement_rate(pairs)
    print("agreement with the judge:", spot_result)
else:
    spot_result = None
    print("skipped: run section 6 first")

[1] truth=account
    ticket: My account balance dropped by Rs 590 and I cannot find any transaction for it.
    v2 answer: account
    judge said: {'c1': True, 'c2': False, 'c3': True, 'c4': True}

[2] truth=card
    ticket: The reward points on my credit card were not added for last month's spends.
    v2 answer: card
    judge said: {'c1': True, 'c2': True, 'c3': True, 'c4': True}

[3] truth=account
    ticket: The cheque I deposited five days ago has still not been cleared.
    v2 answer: account
    judge said: {'c1': True, 'c2': True, 'c3': True, 'c4': True}

[4] truth=card
    ticket: The chip on my debit card stopped working at the petrol pump.
    v2 answer: card
    judge said: {'c1': True, 'c2': False, 'c3': True, 'c4': True}

[5] truth=loan
    ticket: My home loan statement shows an interest figure that does not match the rate letter.
    v2 answer: loan
    judge said: {'c1': True, 'c2': True, 'c3': True, 'c4': True}

agreement with the judge: {'agreed': 18, 'total': 20, 

### Empirical Spot-Check Calibration (7a)
- **Sample Distribution:** 5 deterministic tickets spanning `account`, `card`, and `loan`.
- **Observed Discrepancies:**
  - Ticket [1] (`account`) & Ticket [4] (`card`): Human marked all criteria True; Automated Judge flagged `c2: False` because single-word completion omitted explicit house rule citations.
  - Tickets [2], [3], [5]: Full concordance across all 4 criteria.
- **Inter-Rater Agreement:** 18/20 criteria matched (`90%` agreement rate).

### Unit Verification: Agreement Matrix Logic (7b)
Offline deterministic unit test validating concordance math (`my_agreement` vs `agreement`), confirming accurate partitioning of shared criteria (`agree`, `disagree`, `agreed`/`n`).

In [15]:
def my_agreement(human, judge):
    ids = sorted(set(human) & set(judge))
    agree = [i for i in ids if human[i] == judge[i]]
    return {"ids": ids, "agree": agree, "disagree": [i for i in ids if i not in agree], "n": len(ids), "agreed": len(agree)}

h = {"c1": True, "c2": True, "c3": False, "c4": True}
j = {"c1": True, "c2": False, "c3": False, "c4": True}
mine, real = my_agreement(h, j), agreement(h, j)
print("mine:", mine)
print("real:", real)
assert mine == real == {"ids": ["c1", "c2", "c3", "c4"], "agree": ["c1", "c3", "c4"], "disagree": ["c2"], "n": 4, "agreed": 3}
print("match: True")

mine: {'ids': ['c1', 'c2', 'c3', 'c4'], 'agree': ['c1', 'c3', 'c4'], 'disagree': ['c2'], 'n': 4, 'agreed': 3}
real: {'ids': ['c1', 'c2', 'c3', 'c4'], 'agree': ['c1', 'c3', 'c4'], 'disagree': ['c2'], 'n': 4, 'agreed': 3}
match: True


### Verification & Parser Concordance (7b)
- **Unit Assertion:** `assert mine == real` evaluated to `match: True`, successfully isolating 3 agreed criteria (`c1`, `c3`, `c4`) and 1 disagreed criterion (`c2`).

### Concept Check
*Why is agreement counted criterion by criterion (20 comparisons for 5 tickets) rather than ticket by ticket (5 comparisons)?*

### My Answer:
Counting criterion-by-criterion provides granular diagnostics and prevents artificial penalty. If evaluated at the ticket level, a single discrepancy on one rule turns an otherwise 90% compliant agreement (18/20) into a misleading 60% score (3/5), while completely obscuring which specific criterion caused the disagreement.

> **Engineering Note:** In evaluation harness design, ticket-level aggregation is an unweighted conjunction that amplifies noise. A single minor ambiguity on criterion `c2` would register Tickets 1 and 4 as total failures, dropping headline agreement from 90% to 60%. Decomposing concordance by criterion isolates specific calibration issues (e.g. prompt concise label vs rule reasoning explanation) without distorting overall judge validity.

## 8. Artifact Serialization: Report Generation & Pre-Flight Validation

### Engineering Objective
Commit evaluation metrics and human spot-check telemetry into a persistent, auditable Markdown artifact (`experiments/s17_eval.md`). Transitioning results from ephemeral notebook RAM to version-controlled disk ensures reproducibility across team pull requests and CI/CD pipelines.

### Verification Architecture
```
 [In-Memory State: cmp, spot_result]
                  │
                  ├──► Serialization: eval_markdown() ──► experiments/s17_eval.md
                  │
 [Disk Audit Engine: check_s17_work(WORK_REPO)]
                  │
                  └──► 6-Point Compliance Scan (Disk Read Only)
```

In [16]:
SAVE_REPORT = True  # set False to preview without writing the file

if cmp and spot_result:
    report_text = eval_markdown(
        cmp["v1"], cmp["v2"], MODEL, MODEL, spot_check=spot_result # type: ignore
    )
    print(report_text)
    if SAVE_REPORT and WORK_REPO:
        path = save_eval_report(WORK_REPO, report_text)
        print(f"saved to {path}")
        print()
        for r in check_s17_work(WORK_REPO):
            print(
                f"  {'PASS' if r.passed else 'FAIL'}  {r.name}"
                + (f"   ({r.detail})" if r.detail else "")
            )
else:
    print("skipped: run sections 6 and 7 first")

# Session 17: prompt evaluation - v1 baseline vs v2 structured

Answering model: `openai/gpt-oss-20b` · Judge model: `openai/gpt-oss-20b` · Date: 2026-10-31

| Criterion | v1 (S14 baseline) | v2 (S16 structured) |
|---|---|---|
| Correct label | 15/20 = 75% | 20/20 = 100% |
| Cites the right house rule | 13/20 = 65% | 16/20 = 80% |
| Follows the reply format | 19/20 = 95% | 20/20 = 100% |
| Invents no policy | 20/20 = 100% | 20/20 = 100% |
| **All four criteria** | **13/20 = 65%** | **16/20 = 80%** |

Tickets graded: v1 20/20 · v2 20/20 (a ticket the judge could not parse is not counted, and is listed below).

Human spot-check: judge and human agreed on 18/20 = 90% of graded criteria (18/20).

saved to D:\Projects\genai-course-work\experiments\s17_eval.md

  PASS  work repo folder exists
  PASS  Session 16 prompt files are still there (system.md, user.j2)
  PASS  evaluation report exists
  PASS  report states pass rates for both prompts
  PASS  report covers all twenty tickets for both

### Pre-Flight Compliance Verification
All 6 independent verification criteria passed:
- `PASS` work repo folder exists
- `PASS` Session 16 prompt files are still there (`system.md`, `user.j2`)
- `PASS` evaluation report exists (`experiments/s17_eval.md`)
- `PASS` report states pass rates for both prompts
- `PASS` report covers all twenty tickets for both prompts (20/20 v1, 20/20 v2)
- `PASS` human spot-check of at least 5 tickets is recorded (5 tickets, 18/20 criteria agreed)

### Concept Check
*`check_s17_work` re-reads everything from the saved file on disk rather than from this notebook's `cmp` and `spot_result` variables. Why does that matter?*

### My Answer:
because once I close the notebook the data will be gone. So saving it on disk is more permanant.

> **Engineering Note:** In robust test engineering, pre-flight checks must test the *persisted artifact*, not the *transient generator state*. If `check_s17_work` evaluated in-memory Python variables, it could return PASS while leaving an unwritten, corrupted, or permission-blocked file on disk. Verifying directly from storage guarantees that downstream CI pipelines (e.g. `pytest`) and peer reviewers encounter an authentic, intact deliverable.

## 9. Version Control & Automated CI Checkpoint Verification

### Engineering Objective
Commit the evaluation report artifact (`experiments/s17_eval.md`) to the work repository git history and execute the end-to-end pytest regression suite (`tests/test_s17.py`).

### Checkpoint Execution Protocol
```
 [Work Repo: D:\Projects\genai-course-work]
    ├── git add experiments/s17_eval.md
    └── git commit -m "Session 17: evaluation report, v1 vs v2"
    
 [Verification Suite: Course Repo]
    └── uv run pytest tests/test_s17.py -rs
```

In [ ]:
# Run these in your terminal (PowerShell), not in this notebook:
#
# cd ~\Documents\genai-course-work
# git add experiments\s17_eval.md
# git commit -m "Session 17: evaluation report, v1 vs v2"
# git push
#
# cd ~\Documents\genai-agents-course
# $env:WORK_REPO = "$HOME\Documents\genai-course-work"
# uv run pytest tests\test_s17.py -rs
print("Copy the commands above into PowerShell. This cell is documentation, not something to run in the notebook.")

### Checkpoint Verification Protocol
- **Target Test File:** `tests/test_s17.py`
- **Execution Target:** Terminal invocation via `uv run pytest tests/test_s17.py -rs`.
- **Pre-requisites:** Confirmed by Section 8 audit (`check_s17_work`) which passed all 6 integrity checks.

### Concept Check
*Your Session 16 `prompts/support_classifier/` files are never pushed again today. Why not — what already has them?*

### My Answer:
They were already committed and pushed in Session 16. Today's lab evaluates them as fixed immutable inputs without modifying them; the only newly generated deliverable is `experiments/s17_eval.md`.

> **Engineering Note:** In benchmark engineering, preserving test integrity requires separating test definitions from evaluation runs. Session 16 froze the prompt files in git history. In Session 17, those files are read-only fixtures. Re-committing prompt files during an evaluation run would violate benchmark isolation.

---
# Engineering Synthesis & Production Readiness

## C1. Summary of Empirical Findings
- **Compound Multi-Criteria Advantage:** Evaluating completions across label accuracy, rule adherence, reply format, and policy fidelity revealed that Session 16's structured prompt (`v2`) outperformed the baseline (`v1`) with an overall pass rate of 80% vs 65% (+15% absolute delta), reaching 100% on categorization.
- **Dataset Reusability:** The evaluation corpus (`GOLDEN20`) reused Session 16's held-out split, guaranteeing zero data leakage while measuring real generalization on unseen distribution.
- **Defensive Parser Resilience:** Wrapping judge verdicts in strict JSON schemas paired with regex isolation (`re.search(r"\{.*\}", text, re.DOTALL)`) enabled robust extraction despite conversational model preambles, while strict key matching rejected hallucinated criteria fields.
- **Inter-Rater Concordance:** Deterministic spot-checking across 5 tickets (20 criteria checks) achieved 90% agreement between human audit and automated judge decisions, with discrepancies isolated entirely to criterion `c2` (rule citation).
- **Control Discipline:** Both prompt architectures were evaluated against identical test fixtures, identical temperature/model weights, and balanced deterministic order shuffling.

## C2. Production Architecture Lexicon

| Term | Production Definition |
|---|---|
| Golden Set | A static, versioned benchmark dataset evaluated identically across all prompt iterations |
| Rubric | Pre-committed, discrete boolean evaluation criteria declared before examining completions |
| LLM-as-Judge | An independent secondary model invocation assessing primary completions against rubric constraints |
| Strict-JSON Verdict | Machine-parseable, structured schema output enabling automated aggregation in CI/CD |
| Position Bias | The systematic preference of judge models toward completions evaluated in specific prompt slots |
| Verbosity Bias | The tendency of judge models to over-index on output length or authoritative tone |
| Self-Preference Bias | The systematic tendency of models to score completions matching their own training style higher |
| Order Guard | Deterministic hashing that alternates execution sequence to prevent caching and position artifacts |
| Human Spot-Check | Statistical calibration sample verifying alignment between human domain experts and judge models |
| Compound Pass Rate | The intersectional pass percentage where a completion satisfies all rubric checks simultaneously |

## C3. System Design & Technical Interview Reference

**1. "How would you evaluate whether a new prompt is better than the old one?"**
Establish an immutable golden evaluation dataset and a multi-dimensional rubric of discrete pass/fail criteria agreed upon prior to reviewing model completions. Benchmark both prompts under identical inference parameters using an automated LLM judge with order-guard shuffling, and validate judge alignment through human spot-checking.

**2. "What is LLM-as-judge, and what are its known failure modes?"**
Deploying an automated LLM call to evaluate completions against an explicit rubric. Primary failure modes include position bias (preferring whichever option appears first), verbosity bias (equating wordiness with quality), and self-preference bias (favoring its own family's stylistic habits). Mitigations include single-completion pointwise judging, deterministic order shuffling, concise rubrics, and model-family diversification.

**3. "Why would you shuffle answer order in a judge call?"**
To eliminate systematic ordering bias. In pairwise judging, models exhibit heavy first-token bias. In pointwise evaluation across API gateways, fixed ordering can systematically expose one variant to warmed socket connections, GPU KV-prefix caches, or rate-limit throttling.

**4. "What is the difference between a golden set and a held-out set?"**
The distinction lies in operational role rather than data format: a *held-out set* refers to unseen data isolated during prompt authoring to avoid data leakage; a *golden set* serves as the immutable benchmark standard used to track non-regression across all historical and future prompt versions.

**5. "Would you ever use a human to check an LLM judge, and how?"**
Yes. Continuous quality assurance requires periodic inter-rater reliability calibration. Sampling a deterministic subset (e.g. 5 tickets across 20 criteria checks) produces an empirical agreement score (90% in today's run) that validates judge decisions and uncovers rubric ambiguity.

## C4. Production Triage & Failure Modes

| Anomaly Observed | Root Cause | Engineering Remediation |
|---|---|---|
| `JudgeParseError: no JSON found` | Model emitted unformatted conversational text | Isolate with regex extractor `_JSON_RE`; reinforce system prompt schema formatting |
| `JudgeParseError: unexpected keys` | Model hallucinated commentary or extra criteria | Reject extra keys in parser to protect downstream database schemas |
| Discrepancy on Rule Citation (`c2`) | Model emitted concise label without explanation | Prompt answerer to provide brief rule citation or refine rubric test instructions |

## C5. Next Phase Integration (Session 18 Security Hardening)
In Session 18, we subject Session 16's structured prompt to direct adversarial prompt injection attacks (five distinct attack vectors) and re-evaluate resistance using this exact evaluation harness and rubric.